# 06 — Wider data and scale: multi-variable panels

Real extracts rarely have only two columns. This notebook loads an **8-variable**
synthetic job-training panel, explores it like microdata, and shows that
order-only interventional queries stay tractable because partialid **prunes
inert covariates** from the program. We then look at settings where scale
*does* bite (higher treatment cardinality, multiple causes).

Honesty check: under order-only Manski, the solver is **not** optimizing a full
8-way response-type LP — structure reduces the active program to
`{treated, employed}`. That reduction *is* the product capability you want when
files get wide.


## A wider extract


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import time
import pandas as pd
from partialid import (
    Assumptions, Query, Variables, bound, bound_path, from_counts, from_dataframe,
)

cols = ["site", "age_bin", "educ", "prior_emp", "urban", "spouse", "treated", "employed"]
df = pd.read_csv(tutorial_data("job_training_panel.csv"))
print(f"N = {len(df)}  columns = {list(df.columns)}")
print(df.head(6))
print()
print("missing values:", int(df.isna().sum().sum()))
print(df[cols].mean().round(3).rename("mean"))


N = 5000  columns = ['site', 'age_bin', 'educ', 'prior_emp', 'urban', 'spouse', 'treated', 'employed']
   site  age_bin  educ  prior_emp  urban  spouse  treated  employed
0     0        2     0          0      1       0        0         1
1     0        2     1          1      1       0        0         1
2     2        2     1          1      1       0        1         1
3     1        2     0          1      0       0        0         1
4     1        1     1          1      1       1        1         1
5     1        1     0          1      0       0        1         0

missing values: 0
site         1.009
age_bin      1.002
educ         0.398
prior_emp    0.476
urban        0.591
spouse       0.509
treated      0.507
employed     0.618
Name: mean, dtype: float64


## Look at employment patterns


In [2]:
print("Employment by treatment:")
print(df.groupby("treated")["employed"].mean())
print()
print("Employment by site × treatment:")
print(df.groupby(["site", "treated"])["employed"].mean().unstack("treated").round(3))
print()
print("Employment by educ:")
print(df.groupby("educ")["employed"].mean().round(3))


Employment by treatment:
treated
0    0.506088
1    0.727524
Name: employed, dtype: float64

Employment by site × treatment:
treated      0      1
site                 
0        0.540  0.738
1        0.464  0.734
2        0.513  0.710

Employment by educ:
educ
0    0.567
1    0.695
Name: employed, dtype: float64


## Question on the full 8-column table

Same scientific question as notebook 01, but `from_dataframe` keeps every
column. Causal order: pretreatment covariates, then treatment, then outcome.


In [3]:
data = from_dataframe(df, columns=cols)
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
# Pretreatment → treatment → outcome (explicit order; clearer than manski's treatment-first covariates)
a = Assumptions.order(*cols)
print(Q.describe())
print("variables:", data.variables.names)


P(employed=1 | do(treated=1))
variables: ('site', 'age_bin', 'educ', 'prior_emp', 'urban', 'spouse', 'treated', 'employed')


## Manski on the wide table — watch the prune notes


In [4]:
t0 = time.perf_counter()
r = bound(data.variables, data.P, Q, a)
t1 = time.perf_counter()
print(r.summary(Q))
print(f"elapsed: {t1 - t0:.3f}s")
print("notes (structure / prune):")
for n in r.notes:
    if "inert" in n or "prune" in n or "program:" in n or "c-degree" in n:
        print(" ", n)


query: P(employed=1 | do(treated=1))
interval: [0.369, 0.8618]
width: 0.4928
method: fiber
tight: True
success: True
elapsed: 0.017s
notes (structure / prune):
  inert prune dropped ['site', 'age_bin', 'educ', 'prior_emp', 'urban', 'spouse']; kept ['treated', 'employed']
  program: linear (∑ free-block degrees=1, K=1)
  c-degree-1 single-block LP on ['treated', 'employed']


The note `inert prune dropped [...]` means covariates were removed from the
*program* as irrelevant for this order-only interventional query. The dataset
can stay wide; the LP stays small.


## Equivalence on the collapsed margin


In [5]:
ct = df.groupby(["treated", "employed"]).size()
counts = {k: int(v) for k, v in ct.items()}
data_xy = from_counts(("treated", "employed"), counts)
r_xy = bound(
    data_xy.variables, data_xy.P, Q,
    Assumptions.manski("treated", "employed"),
)
print(f"wide table: [{r.lower:.6f}, {r.upper:.6f}]")
print(f"2×2 margin: [{r_xy.lower:.6f}, {r_xy.upper:.6f}]")
print("match?", abs(r.lower - r_xy.lower) < 1e-8 and abs(r.upper - r_xy.upper) < 1e-8)


wide table: [0.369000, 0.861800]
2×2 margin: [0.369000, 0.861800]
match? True


## Sensitivity path on the wide table


In [6]:
path = bound_path(
    data.variables, data.P, Q, a,
    param="Lambda",
    treatment="treated",
    theta_min=1.0,
    theta_max=4.0,
    n_grid=7,
)
print(path.summary(at=2.0))
for lam in (1.0, 2.0, 3.0, 4.0):
    lo, hi, w = path.at(lam)
    print(f"Λ={lam:g}: [{lo:.4f}, {hi:.4f}]  width={w:.4f}")


sensitivity path in Lambda
range: [1, 4]
closed_form: False
width @ endpoints: -1.11022e-16 → 0.3696
at Lambda=2: [0.593247, 0.794662] (width 0.201415)
Λ=1: [0.7275, 0.7275]  width=0.0000
Λ=2: [0.5932, 0.7947]  width=0.2014
Λ=3: [0.4885, 0.8170]  width=0.3285
Λ=4: [0.4586, 0.8282]  width=0.3696


## Stratum-specific analysis

A large file is not one monolithic question. Slice the cohort when the science
is conditional (e.g. one study site).


In [7]:
df_site = df[df["site"] == 1]
data_site = from_dataframe(df_site, columns=cols)
r_site = bound(data_site.variables, data_site.P, Q, a)
print(f"site==1  N={len(df_site)}")
print(r_site.summary(Q))

# Or use Query.condition on a pretreatment covariate
Q_educ = Query.interventional(
    outcome="employed", do={"treated": 1}, value=1, condition={"educ": 1},
)
r_educ = bound(data.variables, data.P, Q_educ, a)
print(Q_educ.describe())
print(r_educ.summary(Q_educ))


site==1  N=1676
query: P(employed=1 | do(treated=1))
interval: [0.381862, 0.861575]
width: 0.479714
method: fiber
tight: True
success: True
P(employed=1 | do(treated=1)), educ=1
query: P(employed=1 | do(treated=1)), educ=1
interval: [0.459799, 0.869849]
width: 0.41005
method: fiber
tight: True
success: True


## When scale bites: ternary dose

Raising treatment cardinality grows the response-type space. Here a small
ternary exposure table (shown explicitly) still runs quickly, but timings and
widths illustrate the cost of richer treatments.


In [8]:
# Synthetic ternary dose × binary outcome (visible table, not a black-box helper)
import numpy as np
dose_cols = ["dose", "employed"]
# probabilities over dose∈{0,1,2}, employed∈{0,1}
P_dose = {
    (0, 0): 0.18, (0, 1): 0.12,
    (1, 0): 0.10, (1, 1): 0.20,
    (2, 0): 0.15, (2, 1): 0.25,
}
rows = []
rng = np.random.default_rng(3)
keys = list(P_dose)
p = np.array([P_dose[k] for k in keys]); p = p / p.sum()
idx = rng.choice(len(keys), size=2000, p=p)
df_dose = pd.DataFrame([keys[i] for i in idx], columns=dose_cols)
print(pd.crosstab(df_dose["dose"], df_dose["employed"], normalize="index").round(3))

data_dose = from_dataframe(df_dose, columns=dose_cols)
Q_dose = Query.interventional(outcome="employed", do={"dose": 1}, value=1)
t0 = time.perf_counter()
r_dose = bound(
    data_dose.variables, data_dose.P, Q_dose,
    Assumptions.order("dose", "employed") & Assumptions.tan_msm("dose", 2.0),
)
print(r_dose.summary(Q_dose), f"  elapsed={time.perf_counter()-t0:.3f}s")


employed      0      1
dose                  
0         0.613  0.387
1         0.335  0.665
2         0.369  0.631
query: P(employed=1 | do(dose=1))
interval: [0.429729, 0.783479]
width: 0.35375
method: msm_agg/msm_transport_lp
tight: True
success: True
message: ok   elapsed=0.030s


## When scale bites: multi-cause

Two treatments and one outcome under order-only: intervening on one cause
keeps a larger active set than a simple binary XY problem.


In [9]:
V_mc = Variables.from_binary("T0", "T1", "employed")
# Small joint with full support
P_mc = {
    (0, 0, 0): 0.10, (0, 0, 1): 0.15,
    (0, 1, 0): 0.08, (0, 1, 1): 0.12,
    (1, 0, 0): 0.09, (1, 0, 1): 0.14,
    (1, 1, 0): 0.11, (1, 1, 1): 0.21,
}
s = sum(P_mc.values())
P_mc = {k: v / s for k, v in P_mc.items()}
Q_mc = Query.interventional(outcome="employed", do={"T0": 1}, value=1)
t0 = time.perf_counter()
r_mc = bound(V_mc, P_mc, Q_mc, Assumptions.order("T0", "T1", "employed"))
print(r_mc.summary(Q_mc), f"  elapsed={time.perf_counter()-t0:.3f}s")
print("notes:")
for n in r_mc.notes:
    if "inert" in n or "program:" in n or "c-degree" in n:
        print(" ", n)


query: P(employed=1 | do(T0=1))
interval: [0.35, 0.8]
width: 0.45
method: fiber
tight: True
success: True   elapsed=0.001s
notes:
  inert prune dropped ['T1']; kept ['T0', 'employed']
  program: linear (∑ free-block degrees=1, K=1)
  c-degree-1 single-block LP on ['T0', 'employed']


## Practical guidance

1. **Wide binary panels** with order-only interventional queries are fine —
   inert pruning keeps the LP on the active margin.
2. **Higher cardinality** and **multiple active causes** grow cost; start binary
   when you can.
3. Prefer **design structure** (IV, exclusions, observed confounders) over
   brute-force joints.
4. Always check **`tight`**.

**Next (optional):** [07_advanced_identification.ipynb](07_advanced_identification.ipynb)
for program degree, outer approximations, and `prefer_poly`.
